# Step 12: Round 1h Relation-Chain Consolidation Diagnosis

这个 notebook 只做 `Round 1h`：

1. 生成一个更收紧的 revised `relation_chain` consolidation
2. 只在 `wiki_dev_2639` 上比较：
   - `no_memory`
   - original relevant consolidation
   - revised relevant consolidation
   - irrelevant consolidation

固定不变：

- model
- `Round 1b` prompt scaffold
- scoring / answer extraction
- target
- pairing


In [1]:
import csv
import gc
import json
import os
import re
import string
from pathlib import Path

try:
    import torch
except ImportError:
    torch = None

try:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
except ImportError:
    AutoTokenizer = None
    AutoModelForCausalLM = None
    BitsAndBytesConfig = None


## 1. 配置路径、模型与运行参数


In [2]:
PROJECT_ROOT_OVERRIDE = ''

MODEL_ID = 'Qwen/Qwen3.5-9B'
FALLBACK_MODEL_ID = 'Qwen/Qwen3.5-4B'
HF_TOKEN = os.environ.get('HF_TOKEN', '')

RUN_GENERATION = True
USE_4BIT = False
ENABLE_THINKING = False
MAX_NEW_TOKENS = 1600
DO_SAMPLE = False


def candidate_roots():
    candidates = []
    if PROJECT_ROOT_OVERRIDE.strip():
        candidates.append(Path(PROJECT_ROOT_OVERRIDE).expanduser())

    env_root = os.environ.get('SELECT_TRANSFER_ROOT', '').strip()
    if env_root:
        candidates.append(Path(env_root).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd.parent,
        cwd / '2026_SelectTransfer',
        Path('/content/2026_SelectTransfer'),
        Path('/workspace/2026_SelectTransfer'),
        Path('/root/2026_SelectTransfer'),
        Path('/kaggle/working/2026_SelectTransfer'),
    ])

    ordered = []
    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        ordered.append(candidate)
    return ordered


def detect_project_root():
    checked = []
    for candidate in candidate_roots():
        checked.append(str(candidate))
        if candidate.exists() and (candidate / 'pilot').exists() and (candidate / 'results').exists():
            return candidate
    raise FileNotFoundError(
        'Could not locate project root. Set PROJECT_ROOT_OVERRIDE or SELECT_TRANSFER_ROOT to the uploaded 2026_SelectTransfer directory. Checked: ' + ' | '.join(checked)
    )


PROJECT_ROOT = detect_project_root()
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
RESULTS_DIR = PROJECT_ROOT / 'results' / '11_round1h_run'
RAW_OUT_DIR = RESULTS_DIR / 'raw_outputs'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RAW_OUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_SETS_PATH = PROJECT_ROOT / 'pilot' / 'source_sets.csv'
BASE_TAXONOMY_PATH = PROJECT_ROOT / 'pilot' / 'taxonomy.csv'
BATCH2_ANNOTATION_PATH = PROJECT_ROOT / 'results' / '09_relation_chain_bridge_expansion_batch2' / 'candidate_batch2_for_subtype_annotation.csv'
SUBSET_PATH = PROJECT_ROOT / 'results' / '11_round1h_prep' / 'relation_chain_consolidation_subset.csv'

SAMPLED_JSON_PATH = PROJECT_ROOT / 'results' / '01_sampling' / 'sampled_20_full.json'
EXPANDED_JSON_PATH = PROJECT_ROOT / 'results' / '02_hotpotqa_comparison_expansion' / 'candidate_batch_filtered_full.json'
BATCH2_FULL_JSON_PATH = PROJECT_ROOT / 'results' / '09_relation_chain_bridge_expansion_batch2' / 'candidate_batch2_full.json'

ORIG_RELEVANT_CONSOLIDATION_PATH = ARTIFACTS_DIR / 'hp_relation_chain_bridge_set_01' / 'cross_episode_consolidation.md'
IRRELEVANT_CONSOLIDATION_PATH = ARTIFACTS_DIR / 'hp_bridge_set_01' / 'cross_episode_consolidation.md'
REVISED_PROMPT_PATH = RESULTS_DIR / 'revised_relation_chain_consolidation_prompt.md'
REVISED_CONSOLIDATION_PATH = RESULTS_DIR / 'revised_relation_chain_consolidation.md'

REQUIRED_INPUTS = {
    'working source sets': SOURCE_SETS_PATH,
    'working taxonomy': BASE_TAXONOMY_PATH,
    'Batch 2 subtype annotation': BATCH2_ANNOTATION_PATH,
    'Round 1h subset': SUBSET_PATH,
    'Round 1 sampled payload json': SAMPLED_JSON_PATH,
    'comparison expansion payload json': EXPANDED_JSON_PATH,
    'relation-chain Batch 2 payload json': BATCH2_FULL_JSON_PATH,
    'original relevant consolidation': ORIG_RELEVANT_CONSOLIDATION_PATH,
    'irrelevant attribute consolidation': IRRELEVANT_CONSOLIDATION_PATH,
}

missing_inputs = {name: str(path) for name, path in REQUIRED_INPUTS.items() if not path.exists()}
if missing_inputs:
    missing_lines = [f'- {name}: {path}' for name, path in missing_inputs.items()]
    raise FileNotFoundError('Detected project root but some required inputs are missing:\n' + '\n'.join(missing_lines))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RESULTS_DIR =', RESULTS_DIR)
print('MODEL_ID =', MODEL_ID)
print('RUN_GENERATION =', RUN_GENERATION)
print('required inputs checked =', len(REQUIRED_INPUTS))
print('torch import available =', torch is not None)
if torch is not None and torch.cuda.is_available():
    print('CUDA device =', torch.cuda.get_device_name(0))
    print('CUDA bf16 supported =', torch.cuda.is_bf16_supported())
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print('CUDA total memory (GB) =', round(total_gb, 2))


PROJECT_ROOT = /root/2026_SelectTransfer
RESULTS_DIR = /root/2026_SelectTransfer/results/11_round1h_run
MODEL_ID = Qwen/Qwen3.5-9B
RUN_GENERATION = True
required inputs checked = 9
torch import available = True
CUDA device = NVIDIA GeForce RTX 4090
CUDA bf16 supported = True
CUDA total memory (GB) = 47.37


## 2. 读取 source set、taxonomy 和 payload


In [3]:
def read_csv(path):
    with path.open(newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))


def load_json(path):
    return json.loads(path.read_text(encoding='utf-8'))


def parse_members(cell):
    return [x.strip() for x in str(cell).split('|') if x.strip()]


source_set_rows = read_csv(SOURCE_SETS_PATH)
base_taxonomy_rows = read_csv(BASE_TAXONOMY_PATH)
batch2_taxonomy_rows = read_csv(BATCH2_ANNOTATION_PATH)
subset_rows = read_csv(SUBSET_PATH)
subset_ids = [row['target_task_id'] for row in subset_rows]

sampled_rows = load_json(SAMPLED_JSON_PATH)
expanded_rows = load_json(EXPANDED_JSON_PATH)
batch2_rows = load_json(BATCH2_FULL_JSON_PATH)
all_payload_rows = sampled_rows + expanded_rows + batch2_rows
all_payload = {row['task_id']: row for row in all_payload_rows}

taxonomy_map = {}
for row in base_taxonomy_rows:
    taxonomy_map[row['task_id']] = row
for row in batch2_taxonomy_rows:
    taxonomy_map[row['task_id']] = {
        'task_id': row['task_id'],
        'dataset': 'HotpotQA',
        'question': row['question'],
        'answer': row['answer'],
        'reasoning_label': row['reasoning_label'],
        'keep_drop': row['keep_drop'],
        'note': row['note'],
    }

source_set_map = {row['source_set_id']: row for row in source_set_rows}
relation_chain_source = source_set_map['hp_relation_chain_bridge_set_01']
relation_chain_members = parse_members(relation_chain_source['member_task_ids'])

missing_member_payload = [task_id for task_id in relation_chain_members if task_id not in all_payload]
missing_member_taxonomy = [task_id for task_id in relation_chain_members if task_id not in taxonomy_map]
if missing_member_payload or missing_member_taxonomy:
    raise FileNotFoundError('Relation-chain source set inputs are incomplete.')

print('subset target ids =', subset_ids)
print('relation-chain members =', relation_chain_members)
print('payload rows available =', len(all_payload))


subset target ids = ['wiki_dev_2639']
relation-chain members = ['hp_dev_7398', 'hp_dev_2485', 'hp_dev_1892', 'hp_dev_5315', 'hp_dev_7220']
payload rows available = 52


## 3. 构造 revised consolidation prompt


In [4]:
def extract_support_sentences(raw):
    support = raw.get('supporting_facts', {})
    context = raw.get('context', {})
    titles = context.get('title', []) or []
    sentences = context.get('sentences', []) or []

    extracted = []
    for title, sent_id in zip(support.get('title', []), support.get('sent_id', [])):
        sentence_text = ''
        if title in titles:
            idx = titles.index(title)
            title_sents = sentences[idx]
            if 0 <= sent_id < len(title_sents):
                sentence_text = str(title_sents[sent_id]).strip()
        extracted.append({'title': title, 'sent_id': sent_id, 'sentence': sentence_text})
    return extracted


def task_card(task_id):
    taxonomy = taxonomy_map[task_id]
    raw = all_payload[task_id].get('raw') or all_payload[task_id].get('full_example') or all_payload[task_id]
    support_entries = extract_support_sentences(raw)
    support_titles = [entry['title'] for entry in support_entries]
    return {
        'task_id': task_id,
        'dataset': taxonomy.get('dataset', 'HotpotQA'),
        'reasoning_label': taxonomy['reasoning_label'],
        'question': taxonomy['question'],
        'answer': taxonomy['answer'],
        'taxonomy_note': taxonomy['note'],
        'raw_type': raw.get('type', ''),
        'level': raw.get('level', ''),
        'support_titles': support_titles,
        'support_entries': support_entries,
    }


def render_task_card(card):
    lines = [
        f"### {card['task_id']}",
        f"- question: {card['question']}",
        f"- answer: {card['answer']}",
        f"- reasoning_label: {card['reasoning_label']}",
        f"- taxonomy_note: {card['taxonomy_note']}",
        f"- raw_type: {card['raw_type']}",
        f"- level: {card['level']}",
        f"- support_titles: {', '.join(card['support_titles'])}",
        '- minimal_support:'
    ]
    for entry in card['support_entries']:
        lines.append(f"  - [{entry['title']}#{entry['sent_id']}] {entry['sentence']}")
    return '\n'.join(lines)


relation_cards = [task_card(task_id) for task_id in relation_chain_members]
rendered_cards = '\n\n'.join(render_task_card(card) for card in relation_cards)

REVISED_CONSOLIDATION_SYSTEM_PROMPT = """You are revising a reusable memory artifact for an LLM agent experiment.

Your job is to write a tighter cross-episode consolidation for relation-chain bridge tasks.

Hard constraints:
- Output markdown only.
- Keep the same artifact family: shared structure, applicability, operational heuristic, and boundary risk.
- Make the heuristic branch-sensitive rather than generic.
- Prioritize explicit named kinship links already present in context.
- Do not invent evidence that is not present.
- Do not mention this prompt or the experiment instructions.

Important failure to avoid:
- Do not let the artifact drift into abstract family-tree discussion that causes the model to ignore an explicitly named spouse branch already present in context.
- Do not over-emphasize missing-information warnings so strongly that the model stops before checking the spouse branch.
"""

REVISED_CONSOLIDATION_USER_PROMPT = f"""Write a revised `cross_episode_consolidation.md` for the following source set.

Source Set ID: hp_relation_chain_bridge_set_01
Cluster: bridge
Source Set Note: {relation_chain_source['note']}

Input episodes:
{rendered_cards}

Required output structure:

# Cross-Episode Consolidation

## Source Set
- source_set_id: ...
- cluster: ...

## Shared Structure
- 3 to 5 bullets

## Applicability
- when this memory is likely useful
- when it is not the right memory to use

## Operational Heuristic
- a short ordered checklist for applying this memory form
- include an explicit branch-sensitive instruction for in-law questions
- include an explicit warning: do not switch to the subject's own siblings unless the context supports that branch

## Boundary / Failure Risk
- 2 to 3 bullets only
- keep this section shorter than the heuristic

Important distinction:
- this artifact must remain more abstract than episodic_trace
- but it must be operational enough to preserve spouse-branch continuation in in-law questions
- avoid long meta-level genealogy discussion
"""

REVISED_PROMPT_PATH.write_text(REVISED_CONSOLIDATION_USER_PROMPT + '\n', encoding='utf-8')
print('revised prompt ->', REVISED_PROMPT_PATH)


revised prompt -> /root/2026_SelectTransfer/results/11_round1h_run/revised_relation_chain_consolidation_prompt.md


## 4. 本地模型加载与 revised consolidation 生成


In [5]:
def infer_torch_dtype():
    if torch is None:
        raise ImportError('torch is not available.')
    if torch.cuda.is_available():
        if torch.cuda.is_bf16_supported():
            return torch.bfloat16
        return torch.float16
    return torch.float32


def load_local_model(model_id):
    if torch is None or AutoTokenizer is None or AutoModelForCausalLM is None:
        raise ImportError('Required local generation libraries are not available.')

    token = HF_TOKEN or None
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=token)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {'device_map': 'auto', 'token': token}
    torch_dtype = infer_torch_dtype()

    if USE_4BIT:
        if BitsAndBytesConfig is None:
            raise ImportError('BitsAndBytesConfig is not available.')
        compute_dtype = torch_dtype if torch_dtype != torch.float32 else torch.float16
        model_kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,
        )
    else:
        model_kwargs['torch_dtype'] = torch_dtype

    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    model.eval()
    return tokenizer, model, torch_dtype


def build_chat_text(system_prompt, user_prompt):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def get_model_device(model_obj):
    return next(model_obj.parameters()).device


def generate_markdown(system_prompt, user_prompt):
    prompt_text = build_chat_text(system_prompt, user_prompt)
    model_inputs = tokenizer([prompt_text], return_tensors='pt')
    model_device = get_model_device(model)
    model_inputs = {k: v.to(model_device) for k, v in model_inputs.items()}
    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    output_ids = generated_ids[0][model_inputs['input_ids'].shape[1]:]
    text = tokenizer.decode(output_ids, skip_special_tokens=True).strip()
    if not ENABLE_THINKING:
        text = text.replace('<think>', '').replace('</think>', '').strip()
    token_usage = len(model_inputs['input_ids'][0]) + len(output_ids)
    return text, token_usage


tokenizer = None
model = None
torch_dtype = None
revised_generation_status = 'prompt_only'
revised_generation_error = ''
revised_generation_tokens = 0
revised_text = ''
if RUN_GENERATION:
    tokenizer, model, torch_dtype = load_local_model(MODEL_ID)
    print('Loaded model =', MODEL_ID)
    print('Loaded dtype =', torch_dtype)
    try:
        revised_text, revised_generation_tokens = generate_markdown(REVISED_CONSOLIDATION_SYSTEM_PROMPT, REVISED_CONSOLIDATION_USER_PROMPT)
        REVISED_CONSOLIDATION_PATH.write_text(revised_text + '\n', encoding='utf-8')
        revised_generation_status = 'generated'
    except Exception as e:
        revised_generation_error = str(e)[:300]
        revised_generation_status = 'error'
else:
    REVISED_CONSOLIDATION_PATH.write_text('', encoding='utf-8')

print('revised consolidation status =', revised_generation_status)
print('revised consolidation path =', REVISED_CONSOLIDATION_PATH)
if revised_generation_error:
    print('generation error =', revised_generation_error)


`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Loaded model = Qwen/Qwen3.5-9B
Loaded dtype = torch.bfloat16
revised consolidation status = generated
revised consolidation path = /root/2026_SelectTransfer/results/11_round1h_run/revised_relation_chain_consolidation.md


## 5. 使用 Round 1b scaffold 运行单目标对照


In [6]:
def build_context_paragraphs(raw_context):
    context = raw_context.get('context', {})

    if isinstance(context, dict):
        titles = context.get('title', []) or []
        sentences = context.get('sentences', []) or []
        paragraphs = []
        for title, sents in zip(titles, sentences):
            text = ' '.join(str(s) for s in sents)
            paragraphs.append(f"### {title}\n{text}")
        return '\n\n'.join(paragraphs)

    if isinstance(context, str):
        try:
            parsed = json.loads(context)
        except json.JSONDecodeError:
            return context.strip()

        paragraphs = []
        for item in parsed:
            if not isinstance(item, list) or len(item) != 2:
                continue
            title, sents = item
            text = ' '.join(str(s) for s in sents)
            paragraphs.append(f"### {title}\n{text}")
        return '\n\n'.join(paragraphs)

    return str(context)


def assemble_prompt(target_task, condition_label, artifact_content=None):
    context = build_context_paragraphs(target_task['raw'])
    question = target_task['question']

    base_header = (
        'You are a question-answering agent. '
        'Your task is to answer a multi-hop reasoning question '
        'using the provided context paragraphs.'
    )

    context_section = f"## Context\n\n{context}"
    question_section = f"## Question\n\n{question}"

    if condition_label == 'no_memory':
        memory_section = ''
    else:
        memory_section = (
            '## Past Experience\n\n'
            'The following notes summarize patterns from previously solved tasks '
            'that may or may not be relevant to the current question. '
            'Use them only if they help your reasoning — do not force-apply them.\n\n'
            f'{artifact_content}'
        )

    instructions_section = (
        '## Instructions\n\n'
        '- Read all context paragraphs carefully.\n'
        '- Work through the reasoning chain explicitly before deciding the answer.\n'
        '- In ## Reasoning, write 3 to 6 short bullet points grounded in the provided context.\n'
        '- If past experience is shown, either use it explicitly or state briefly why it is not useful here.\n'
        '- Keep the reasoning concise and evidence-grounded.\n'
        '- In ## Final Answer, give only the final short answer phrase.'
    )

    parts = [base_header, '', context_section, '', question_section]
    if memory_section:
        parts += ['', memory_section]
    parts += ['', instructions_section, '', '## Reasoning', '', '## Final Answer']
    return '\n\n'.join(parts)


def extract_final_answer(model_output):
    if '## Final Answer' in model_output:
        tail = model_output.split('## Final Answer')[-1].strip().splitlines()
        for line in tail:
            line = line.strip()
            if line:
                return line
        return ''
    lines = [l.strip() for l in model_output.strip().split('\n') if l.strip()]
    return lines[-1] if lines else ''


def reasoning_present(model_output):
    return int('## Reasoning' in model_output)


def final_answer_present(model_output):
    return int('## Final Answer' in model_output)


def memory_reference_type(model_output):
    low = model_output.lower()
    if 'past experience' in low and ('not useful' in low or 'not relevant' in low or 'ignore' in low):
        return 'explicit_reject'
    if 'past experience' in low or 'the notes' in low or 'the memory' in low or 'based on the pattern' in low:
        return 'explicit_use'
    return 'implicit_or_none'


def normalize_answer(text):
    text = text.lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = ' '.join(text.split())
    return text


def compute_em(pred, gold):
    return int(normalize_answer(pred) == normalize_answer(gold))


def compute_f1(pred, gold):
    pred_tokens = set(normalize_answer(pred).split())
    gold_tokens = set(normalize_answer(gold).split())
    if not pred_tokens or not gold_tokens:
        return 0.0
    common = pred_tokens & gold_tokens
    if not common:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def build_chat_text_for_run(prompt_text):
    messages = [{'role': 'user', 'content': prompt_text}]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_local(prompt_text):
    chat_text = build_chat_text_for_run(prompt_text)
    model_inputs = tokenizer([chat_text], return_tensors='pt')
    model_device = get_model_device(model)
    model_inputs = {k: v.to(model_device) for k, v in model_inputs.items()}
    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    output_ids = generated_ids[0][model_inputs['input_ids'].shape[1]:]
    text = tokenizer.decode(output_ids, skip_special_tokens=True).strip()
    if not ENABLE_THINKING:
        text = text.replace('<think>', '').replace('</think>', '').strip()
    token_usage = len(model_inputs['input_ids'][0]) + len(output_ids)
    return text, token_usage


target_task = all_payload['wiki_dev_2639']
revised_consolidation_text = REVISED_CONSOLIDATION_PATH.read_text(encoding='utf-8') if REVISED_CONSOLIDATION_PATH.exists() else ''
original_relevant_consolidation = ORIG_RELEVANT_CONSOLIDATION_PATH.read_text(encoding='utf-8')
irrelevant_consolidation = IRRELEVANT_CONSOLIDATION_PATH.read_text(encoding='utf-8')

run_plan = [
    {'run_id': 'r1h_no_memory_wiki_dev_2639', 'condition': 'no_memory', 'source_set_id': 'none', 'artifact_content': None},
    {'run_id': 'r1h_original_relevant_consolidation_wiki_dev_2639', 'condition': 'original_relevant_consolidation', 'source_set_id': 'hp_relation_chain_bridge_set_01', 'artifact_content': original_relevant_consolidation},
    {'run_id': 'r1h_revised_relevant_consolidation_wiki_dev_2639', 'condition': 'revised_relevant_consolidation', 'source_set_id': 'hp_relation_chain_bridge_set_01', 'artifact_content': revised_consolidation_text},
    {'run_id': 'r1h_irrelevant_consolidation_wiki_dev_2639', 'condition': 'irrelevant_consolidation', 'source_set_id': 'hp_bridge_set_01', 'artifact_content': irrelevant_consolidation},
]

results = []
for run in run_plan:
    prompt = assemble_prompt(target_task, run['condition'], run['artifact_content'])
    raw_output = ''
    pred_answer = ''
    token_usage = 0
    failure_status = 'ok'

    if run['condition'] == 'revised_relevant_consolidation' and revised_generation_status != 'generated':
        failure_status = f'revised_artifact_not_ready: {revised_generation_status}'
    elif RUN_GENERATION:
        try:
            raw_output, token_usage = generate_local(prompt)
            pred_answer = extract_final_answer(raw_output)
        except Exception as e:
            failure_status = f'error: {str(e)[:200]}'
    else:
        failure_status = 'dry_run'

    gold = target_task['answer']
    em = compute_em(pred_answer, gold) if pred_answer else 0
    f1 = round(compute_f1(pred_answer, gold), 4) if pred_answer else 0.0
    rp = reasoning_present(raw_output) if raw_output else 0
    fp = final_answer_present(raw_output) if raw_output else 0
    mrt = memory_reference_type(raw_output) if raw_output else 'implicit_or_none'
    parse_success = int(bool(pred_answer))

    results.append({
        'run_id': run['run_id'],
        'target_task_id': 'wiki_dev_2639',
        'condition': run['condition'],
        'source_set_id': run['source_set_id'],
        'em': em,
        'f1': f1,
        'token_usage': token_usage,
        'failure_status': failure_status,
        'pred_answer': pred_answer,
        'gold_answer': gold,
        'prompt_chars': len(prompt),
        'reasoning_present': rp,
        'final_answer_present': fp,
        'memory_reference_type': mrt,
        'parse_success': parse_success,
        'raw_output': raw_output,
        'prompt_text': prompt,
    })

    if raw_output or failure_status == 'dry_run':
        raw_path = RAW_OUT_DIR / f"{run['run_id']}.md"
        with raw_path.open('w', encoding='utf-8') as f:
            f.write(f"# {run['run_id']}\n\n")
            f.write(f"- target_task_id: wiki_dev_2639\n")
            f.write(f"- condition: {run['condition']}\n")
            f.write(f"- source_set_id: {run['source_set_id']}\n")
            f.write(f"- gold_answer: {gold}\n")
            f.write(f"- pred_answer: {pred_answer}\n")
            f.write(f"- em: {em}\n")
            f.write(f"- f1: {f1}\n")
            f.write(f"- token_usage: {token_usage}\n")
            f.write(f"- prompt_chars: {len(prompt)}\n")
            f.write(f"- reasoning_present: {rp}\n")
            f.write(f"- final_answer_present: {fp}\n")
            f.write(f"- memory_reference_type: {mrt}\n")
            f.write(f"- parse_success: {parse_success}\n")
            f.write(f"- failure_status: {failure_status}\n\n")
            f.write('---\n\n')
            f.write(f"## Prompt\n\n```\n{prompt}\n```\n\n")
            f.write(f"## Raw Model Output\n\n```\n{raw_output}\n```\n")

print('total runs =', len(results))
for r in results:
    print(r['run_id'], r['failure_status'], r['pred_answer'])


total runs = 4
r1h_no_memory_wiki_dev_2639 ok Henry Pelham
r1h_original_relevant_consolidation_wiki_dev_2639 ok *   There is no mention of siblings.
r1h_revised_relevant_consolidation_wiki_dev_2639 ok Cannot be determined from the provided context.
r1h_irrelevant_consolidation_wiki_dev_2639 ok Cannot be determined from the provided context.


## 6. 写出结果文件


In [7]:
detail_fieldnames = [
    'run_id', 'target_task_id', 'condition', 'source_set_id',
    'em', 'f1', 'token_usage', 'failure_status',
    'pred_answer', 'gold_answer', 'prompt_chars', 'reasoning_present',
    'final_answer_present', 'memory_reference_type', 'parse_success', 'raw_output'
]

detail_path = RESULTS_DIR / 'round1h_consolidation_results_detail.csv'
with detail_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=detail_fieldnames)
    writer.writeheader()
    for r in results:
        writer.writerow({k: r[k] for k in detail_fieldnames})
print('detail ->', detail_path)

summary_fieldnames = [
    'run_id', 'target_task_id', 'condition', 'source_set_id',
    'em', 'f1', 'token_usage', 'failure_status',
    'reasoning_present', 'final_answer_present', 'memory_reference_type', 'parse_success'
]
summary_path = RESULTS_DIR / 'round1h_consolidation_results.csv'
with summary_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=summary_fieldnames)
    writer.writeheader()
    for r in results:
        writer.writerow({k: r[k] for k in summary_fieldnames})
print('summary ->', summary_path)


detail -> /root/2026_SelectTransfer/results/11_round1h_run/round1h_consolidation_results_detail.csv
summary -> /root/2026_SelectTransfer/results/11_round1h_run/round1h_consolidation_results.csv


## 7. Quick sanity check


In [8]:
for r in results:
    print(r['run_id'], r['em'], r['failure_status'], r['memory_reference_type'])

if model is not None:
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()


r1h_no_memory_wiki_dev_2639 1 ok implicit_or_none
r1h_original_relevant_consolidation_wiki_dev_2639 0 ok explicit_use
r1h_revised_relevant_consolidation_wiki_dev_2639 0 ok implicit_or_none
r1h_irrelevant_consolidation_wiki_dev_2639 0 ok implicit_or_none
